# Bayes Rules

## Chapter 7. MCMC Under The Hood

Code adapted from: https://github.com/pymc-devs/pymc-resources/tree/main/Bayes_Rules

Antonio Esteves @ UMinho

![](BayesRules-book-cover.png)

In this chapter, we will focus our attention on the Metropolis-Hastings algorithm. The main golas are:
* Build a strong conceptual understanding of how Markov chain algorithms work.
* Explore the foundational Metropolis-Hastings algorithm.
* Implement the Metropolis-Hastings algorithm in the Normal-Normal and Beta-Binomial settings for likelihood-prior.

In [ ]:
import numpy    as np
import pandas   as pd
import pymc     as pm
import pyreadr
import requests
import seaborn  as sns
import arviz    as az
import matplotlib.pyplot as plt

from plotnine import (
    aes,
    after_stat,
    geom_histogram,
    geom_line,
    ggplot,
    labs,
    stat_function,
)
from scipy.stats import beta, binom, norm, uniform

In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext watermark
%load_ext nb_black

RANDOM_SEED = 1301

np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

### 7.1 The big idea

Consider a Normal-Normal model with numerical outcome $Y$ that varies normally around an unknown mean $\mu$ with a standard deviation of 0.75:

$Y \mid \mu \sim N(\mu, 0.75)$ <P>
$\mu \sim N(0, 1)$ <P>

The corresponding likelihood $L(\mu \mid y)$ (or $p(y \mid \mu)$) of the data $y$ and the prior for parameter $\mu$, where $y \in (−\infty,\infty)$ and $\mu \in (−\infty,\infty)$, are:

$L(\mu \mid y)=\frac{1}{\sqrt{2\pi 0.75^2}} e^{-\frac{(y-\mu)^2}{2\ 0.75^2}}$ <P>
$f(\mu) =\frac{1}{\sqrt{2 \pi}} e^{-\frac{\mu^2}{2}}$ <P>

Suppose we observe an outcome $Y$=6.25. Then, the posterior model of $\mu$ is Normal with mean 4 and standard deviation 0.6: $\mu \mid (Y= 6.25) \sim N(4,0.6)$.

As in Chapter 6, if we were not able to specify this posterior model for $\mu$, we could approximate it using MCMC simulation. To get a sense for how this works, consider the results of $N$=5000 iterations from a MCMC simulation (Figure 7.1). It helps to think of the illustrated Markov chain {$\mu^{(1)}, \mu^{(2)}, \dots, \mu^{(N)}$} as path taken across the range of posterior plausible values of $\mu$. The left panel trace plot illustrates the path followed
or the sequence of points $\mu^{(i)}$ where we stoped. The histogram on the right panel illustrates the relative
amount of time we spent in each $\mu$ region, or number of times we visited each point, throughout the navigation.

![](../fig/fig-7-01-mcmc-trace-plot-hist.png) <P>
_Figure 7.1_

It is our job, as navigators accros the posterior, to ensure that the number of revisits (density) of each point (value) of $\mu$ is proportional to its posterior plausibility. That is, the chain should spend more time om values of $\mu$ between 2 and 6, where the Normal posterior PDF is higher, and less time visiting $\mu$ values less than 2 or greater than 6, where the posterior is small. This consideration is crucial to producing a collection of points (values) that accurately approximate the posterior, as does the navigation path illustrated in right panel of Figure 7.1.

We can automate building the posterior navigation path using the Metropolis-Hastings algorithm. This algorithm iterates through a two-step process. Assuming the Markov chain is at location $\mu^{(i)} = \mu$ at iteration or navigation point $i$, the next visited point $\mu^{(i+1)}$ is selected as follows:

* Step 1: Propose a random location, $mu'$, for the next visiting point.
* Step 2: Decide whether to go to the proposed location ($\mu^{(i+1)}= \mu'$) or to stay at the current location for another iteration ($\mu^{(i+1)}= \mu$).

If we do not apply any constraints to accepting the proposed new location, meaning that we accept always the proposed location, our navigation path builds a uniform posterior and the algorithm behind the navigation process is called Monte Carlo.

The Monte Carlo algorithm has many use cases and we can implement it simply using a Normal distribution to draw our samples of the posterior. The result is a nice independent sample from the posterior which, in turn, produces
an accurate posterior approximation.

In [ ]:
# Generate 5000 samples from a normal distribution N(mean=4,sigma=0.6)

mean    = 4
std_dev = 0.6

fig, ax = plt.subplots(1, 1)

# Display the probability density function 

x = np.linspace(mean-3.5*std_dev,mean+3.5*std_dev, 100)
ax.plot(x, norm.pdf(x, loc=mean, scale=std_dev), 'r-', lw=5, alpha=0.6, label='pdf')

# generate 5000 random numbers from N(mean,std_dev)

sample = norm.rvs(size=5000, loc=mean, scale=std_dev)

# plot the sample histogram

ax.set_xlim([x[0], x[-1]])
sns.histplot(ax=ax, data=sample, stat='density', bins=16, discrete=False, kde=False,
            facecolor='#0055AA', edgecolor='#0055AA', label='sample hist.')
ax.set_ylabel('f($\mu$)', fontsize=11)
ax.set_xlabel(r'$\mu$', fontsize=11)
fig.suptitle(f'Normal distribution N($\mu$={mean},$\sigma$={std_dev})')
plt.legend(loc='upper right', fontsize= 12)
plt.show()

But we only need MCMC to approximate a Bayesian posterior when that posterior is too complicated to specify. And if a posterior is too complicated to specify, it is typically too complicated to directly sample or draw from as we did in our random generation from the Normal distribution, which mimics a simple Monte Carlo navigation path. This is where the more general Metropolis-Hastings MCMC algorithm comes in. Metropolis-Hastings relies on the fact that, even if we do not know the posterior model, we do know that the posterior PDF is proportional to the product of the known prior PDF and the likelihood function:<P>

$f(\mu \mid y=6.25) \propto f(\mu) \times L(\mu \mid y=6.25)$

This unnormalized posterior is not properly scaled to integrate to 1, but it preserves the shape, central tendency, and variability of the actual posterior.

The <U>first step</U> of the **Metropolis-Hastings** algorithm relies on the fact that, even when we do not know and thus we ca not sample from the posterior model, we can propose Markov chain navigation paths by sampling from a different and more convenient model. As one of many options here, we will utilize a Uniform proposal model with half-width $w$. Specifically, let $\mu^{(i)} = \mu$ denote the current navigation location. Conditioned on this current location, we propose the next location by taking a random draw $\mu'$ from the Uniform model which is centered at the current location $\mu$ and ranges from $\mu-w$ upto $\mu+w$:

$\mu' \mid \mu \sim Uniform (\mu-w, \mu+w)$

with a height (density) equal to

$q(\mu' \mid \mu) = \frac{1}{2w}$ for $\mu \in [\mu-w, \mu+w]$.

Using this method, proposals for $\mu'$ are equally likely to be any value between $\mu-w$ and $\mu+w$.

Figure 7.5 illustrates this idea in a specific scenario. Suppose we are utilizing a Uniform half-width of $w=1$ and that the Markov chain path is at location $\mu=3$. Conditioned on this current location, we will then propose the next location by taking a random draw from the $Uniform(3-w,3+w)=Uniform(2,4)$ model. Thus, the chosen half-width $w=1$ plays an important role here, defining the neighborhood of potential proposals. Specifically, the proposed next location is equally likely to be anywhere within the restricted neighborhood ranging from 2 to 4, around the chain's current location of 3.

![](../fig/fig-7-05-mcmc-metropolis-hastings1.png) <P>
_Figure 7.5_

We may ask, how can proposals drawn from a uniform model produce a decent approximation of the Normal posterior model?Well, they are only proposals that can be rejected or accepted. Mainly, if a proposed location $\mu'$ is "bad", we can reject it. When we do, the chain remains at its current location $\mu$ for at least another iteration.

<U>Step 2</U> of the Metropolis-Hastings algorithm provides a formal process for deciding whether to accept or reject a proposal. Let us first check our intuition about how this process should work. Revisiting Figure 7.5, suppose that our random $Uniform(2,4)$ draw proposes that the chain move from its current location of 3 to 3.8. Does this proposal seem desirable to you? Well, sure. Notice that the unnormalized posterior plausibility of 3.8 is greater than that of 3.
Thus, we want our Markov chain tour to spend more time exploring values of $\mu$ around 3.8 than around 3. Accepting the proposal gives us the chance to do so. In contrast, if our random $Uniform(2,4)$ draw proposed that the chain move from 3 to 2.1, a location with very low posterior plausibility, we might be more hesitant. Consider three possible rules for automating Step 2 in the following quiz.

**Quiz**

Suppose we start our Metropolis-Hastings Markov chain tour at location $\mu^{(1)}=3$ and utilize a Uniform proposal model in Step 1 of the algorithm. Consider three possible rules to follow in Step 2, deciding whether or not to
accept a proposal:<P>
* Rule 1: Never accept the proposed location.
* Rule 2: Always accept the proposed location.
* Rule 3: Only accept the proposed location if its unnormalized posterior plausibility is greater than that of the current location.

Each rule was used to generate one of the Markov chain tours in Figure 7.6. Match each rule to the correct tour.

![](../fig/fig-7-06-mcmc-metropolis-hastings2.png)<P>
_Figure 7.6_

Answer:<P>
  Tour 2 <- rule 1<P> 
  Tour 1 <- rule 3<P>
  Tour 3 <- rule 2<P>

The quiz above presented three poor options for determining whether to accept or reject proposed tour stops in Step 2 of the Metropolis-Hastings algorithm. **Rule 1** presents one extreme: never accept a proposal. This is a terrible idea. It results in the Markov chain remaining at the same location at every iteration (Tour 2), which would certainly produce a silly posterior approximation. **Rule 2** presents the opposite extreme: always accept a proposal. This results in a Markov chain which is not at all discerning in where it travels (Tour 3), completely ignoring the information we have from the unnormalized posterior model regarding the plausibility of a proposal. For example, Tour 3 spends the majority of its time exploring posterior values $\mu$ above 6, which we know to be implausible. **Rule 3** might seem like a reasonable balance between the two extremes: it neither always rejects nor always accepts proposals. However, it is still problematic. Since this rule only accepts a proposed location if its posterior plausibility is greater than that at the current location, it ends up producing a Markov chain similar to that of Tour 1 above. Though this chain oats toward values near $\mu=4$, where the unnormalized posterior PDF is greatest, it then gets stuck there forever.

Putting all of this together, we are closer to understanding how to make the Metropolis-Hastings algorithm work. Upon proposing a navigation location (Step 1), the process for rejecting or accepting this proposal (Step 2) must embrace the
idea that the chain should spend more time exploring areas of high posterior plausibility but should not get stuck there forever:

* **Step 1**: Propose a location, $\mu'$, for the next navigation position by taking a draw from a proposal model.
* **Step 2**: Decide whether to go to the proposed location ($\mu^{(i+1)} = \mu'$) or to stay at the current location for another iteration ($\mu^{(i+1)} = \mu$) as follows:
    * If the unnormalized posterior plausibility of the proposed location $\mu'$ is greater than that of the current location $\mu$, $f(\mu') \times L(\mu' \mid y) > f(\mu) \times L(\mu \mid y)$, definitely go there.
    * Otherwise, maybe go there.

We will see nest what does it mean to "maybe" accept a proposal.

## 7.2 The Metropolis-Hastings algorithm

The Metropolis-Hastings algorithm for constructing a Markov chain path {$\mu^{(1)}, \mu^{(2)}, \dots, \mu^{(N)}$} is formalized here. We will break down the details below, exploring how to implement this algorithm in the context of our Normal-Normal Bayesian model.

**Metropolis-Hastings algorithm**

Conditioned on  data $y$, let parameter $\mu$ have posterior PDF $f(\mu \mid y) \propto f(\mu) \times L(\mu \mid y)$. A Metropolis-Hastings Markov chain for $f(\mu \mid y)$, {$\mu^{(1)}, \mu^{(2)}, \dots, \mu^{(N)}$}, evolves as follows. Let $\mu^{(i)} = \mu$ be the chain's location at iteration $i \in \{1,2, \dots, N-1\}$ and identify the next location
$\mu^{(1i+1)}$ through a two-step process:<P>

* **Step 1: Propose a new location**.<P>
Conditioned on the current location $\mu$, draw a location $\mu'$ from a proposal model with PDF $q(\mu' \mid \mu)$.<P>

* **Step 2: Decide whether or not to go there**.<P>
    * Calculate the acceptance probability, i.e., the probability of accepting the proposal $\mu'$:

    $\alpha = min \{ 1, \frac{f(\mu') \times L(\mu' \mid y)}{f(\mu) \times L(\mu \mid y)} \frac{q(\mu \mid \mu')}{q(\mu' \mid \mu)} \}$ (eq. 7.3)

    * Figuratively, flip a biased coin that has a probability $\alpha$ of getting Heads and a probability $1-\alpha$ of getting Tails. So, if we decide about accept/reject the proposed location $\mu^{(i-1)}$ based of the outcome of the coin flip, we go to the proposed location $\mu'$ with a probability $\alpha$ and we stay at location $\mu$ with a probability $1-\alpha$.

Though the notation and details are new, the algorithm above matches the concepts we developed in the previous section. First, recall that for our Normal-Normal simulation, we utilized a $\mu' \mid \mu \sim Uniform(\mu-w, \mu+w)$ proposal model in Step 1. In fact, this is a bit lazy. Though our Normal-Normal posterior model is defined for $\mu \in (-\infty, \infty)$, the Uniform proposal model lives on a truncated neighborhood around the current chain location. However, utilizing a Uniform proposal model simplifies the Metropolis-Hastings algorithm by the fact that it is **symmetric**. This symmetry exhibits itself in the plot of the Uniform pdf, as well as numerically - the conditional PDF of $\mu'$ given $\mu$ is equivalent to that of $\mu$ given $\mu'$:

$q(\mu' \mid \mu) = q(\mu \mid \mu')$ is equal to $\frac{1}{2w}$ when $\mu$ and $\mu'$ are within $w$ units of each other, or it is equal 0 otherwise.

This symmetry means that the chance of proposing a chain move from $\mu$ to $\mu'$ is the same as proposing a move from $\mu'$ to $\mu$. For example, the Uniform model with a half-width of 1 is equally likely to propose a move from $\mu$=3 to $\mu'$=3.8 as a move from $\mu$=3.8 to $\mu'$=3. We refer to this special case of the Metropolis-Hastings algorithm as simply the Metropolis algorithm.

**Metropolis algorithm**

The Metropolis algorithm is a special case of the Metropolis-Hastings in which the proposal model is symmetric. That is, the chance of proposing a move to $\mu'$ from $\mu$ is equal to that of proposing a move to $\mu$ from $\mu'$:
$q(\mu' \mid \mu) = q(\mu \mid \mu')$. Thus, the acceptance probability simplifies to:

$\alpha = min \{ 1, \frac{f(\mu') \times L(\mu' \mid y)}{f(\mu) \times L(\mu \mid y)} \}$

Inspecting previous equation reveals the nuances of Step 2, determining whether to accept or reject the proposed location drawn in Step 1. First, notice that we can rewrite the acceptance probability $\alpha$ by dividing both the numerator and denominator by $f(y)$:

$\alpha = min \{ 1, \frac{(f(\mu') \times L(\mu' \mid y))/f(y)}{(f(\mu) \times L(\mu \mid y))/f(y)} \} =  min \{ 1, \frac{f(\mu' \mid y)}{f(\mu \mid y)} \}$

This rewrite emphasizes that, though we ca not calculate the posterior PDFs of $\mu'$ and $\mu$, $f(\mu' \mid y)$ and $f(\mu \mid y)$, their ratio is equivalent to that of the unnormalized posterior PDFs (which we can calculate). Thus, the probability of accepting a move from a current location $\mu$ to a proposed location $\mu'$ comes down to a
comparison of their posterior plausibility: $f(\mu' \mid y)$ vs. $f(\mu \mid y)$. There are two possible scenarios here:<P>

* Scenario 1: $f(\mu' \mid y) \geq f(\mu \mid y)$. When the posterior plausibility of $\mu'$ is at least as great as that of $\mu$, $\alpha=1$. Thus, we will definitely move there.
* Scenario 2: $f(\mu' \mid y) < f(\mu \mid y)$. If the posterior plausibility of $\mu'$ is less than that of $\mu$, then $\alpha= \frac{f(\mu' \mid y)}{f(\mu \mid y)} < 1$.

    Thus, we might move there. Further, $\alpha$ approaches 1 as $f(\mu' \mid y)$ gets closer to $f(\mu \mid y)$. That is, the probability of accepting the proposal increases with the plausibility of $\mu'$ relative to $\mu$.

Scenario 1 is straightforward. We will always jump to a more plausible posterior location. To wrap our minds around Scenario 2, a little simulation is helpful. For example, suppose our Markov path is currently at location "3".

In [ ]:
current = 3

Further, suppose we are utilizing a Uniform proposal model with half-width $w=1$. Then to determine the location where we step next, we first propose a location by taking a random draw from the $Uniform(current-1,current+1)$ model (Step
1):

In [ ]:
proposal = uniform.rvs(loc=current-1, scale=2, size=1, random_state=103)

proposal = proposal[0]
print(proposal)

Considering the unnormalized posterior $N(mean=4,sigma=0.6)$, the plausibility of the proposed location (2.406) is quite smaller than that of the current location (3). We can calculate the unnormalized posterior plausibility of these
two $\mu$ locations, $f(\mu)L(\mu \mid y=6.25)$, using the `norm` distribution to evaluate both values:

In [ ]:
proposal_prior_x_likelih = norm.pdf(proposal, 0, 1) * norm.pdf(6.25, proposal, 0.75)
print(proposal_prior_x_likelih)

current_prior_x_likelih = norm.pdf(current, 0, 1) * norm.pdf(6.25, current, 0.75)
print(current_prior_x_likelih)

* `norm.pdf(proposal,0,1)` calculates the $N(0,1)$ prior density $f(\mu')$ at the proposed $\mu'$ value..<P>
* `norm.pdf(6.25, proposal, 0.75)` calculates the Normal likelihood function $L(\mu' \mid y=6.25)$, with $Y=6.25$, standard deviation equal to 0.75 and an unknown mean $\mu$.
  
It follows that the probability $\alpha$ of accepting and subsequently moving to the proposed location is relatively high:

In [ ]:
alpha = min(1, proposal_prior_x_likelih / current_prior_x_likelih)
print(alpha)

To make the final determination, we set up a biased coin which accepts the proposal with probability $\alpha=0.668$ and rejects the proposal with probability $1-\alpha=0.332$. In a random flip of this coin using the `np.random.choice` method, we accept the proposal, meaning that the next_stop on the tour is 2.933:

In [ ]:
next_stop = np.random.choice([proposal, current], size=1, replace=True, p=[alpha, 1-alpha])
next_stop = next_stop[0] 

print(f'{current} -> {proposal} ?')
print(next_stop)

This is merely one of countless possible outcomes for a single iteration of the Metropolis-Hastings algorithm for our Normal posterior. To streamline this process, we will write a function, `one_mh_iteration`, which implements a single Metropolis-Hastings iteration starting from any given current position and utilizing a Uniform proposal model with any given half-width $w$. `one_mh_iteration` implements the same steps as above and returna 3 values: the proposal, the acceptance probability alpha, and next location.

In [ ]:
def one_mh_iteration(w, current):
    proposal       = rng.uniform(low=current - w, high=current + w)
    proposal_p_x_l = norm.pdf(proposal, 0, 1) * norm.pdf(6.25, proposal, 0.75)
    current_p_x_l  = norm.pdf(current, 0, 1) * norm.pdf(6.25, current, 0.75)
    alpha          = min(1, proposal_p_x_l / current_p_x_l)
    next_stop      = np.random.choice(
        [proposal, current], size=1, replace=True, p=[alpha, 1 - alpha]
    )
    next_stop = next_stop[0]
    data      = {"proposal": proposal, "alpha": alpha, "next_stop": next_stop}
    return data

Let us try running `one_mh_iteration` considering the current location equal to 3.

In [ ]:
result = one_mh_iteration(w=1, current=3)
print(result)

We proposed as next location 2.06, which has a low acceptance probability of 0.021. This makes sense since the the posterior density of 2.06 is much lower than that of our current location of 3. Though we do want to explore such extreme values, we do not want to do so often. In fact, we see that upon the flip of our coin, the proposal was rejected and the tour again visits location 3 on its next iteration.

## 7.3 Implementing the Metropolis-Hastings

We have spent much energy understanding how to implement one iteration of the Metropolis-Hastings algorithm. That was the hard part! To construct an entire Metropolis-Hastings navigation path of our $N(4, 0.6)$ posterior, we now just have to repeat this process over and over again. To this end, the `mh_path` function below constructs a Metropolis-Hastings path of any given length $N$, utilizing a Uniform proposal model with any given half-width $w$.

In [ ]:
def mh_path(N, w):
    current = 3
    mu      = np.zeros(N)
    for step in range(N):
        result   = one_mh_iteration(w=w, current=current)
        mu[step] = result['next_stop']
        current  = result['next_stop']
    return pd.DataFrame(mu, columns=["mu"])

Let us make a Markov chain with length 5. 

In [ ]:
mh_path(N=5, w=1)

Let us use `mh_path` to simulate a Markov chain of length $N$=5000 utilizing a Uniform proposal model with half-width $w$=1.

In [ ]:
mh_simulation = mh_path(N=5000, w=1)

A trace plot of the Markov chain path is shown below.

In [ ]:
def plot_trace(
    y:       pd.DataFrame,
    x_label: str,
    y_label: str,
    title:   str):

    N = y.shape[0]

    fig, ax = plt.subplots(1, 1)

    x = np.zeros(N)
    for i in range(N):
        x[i] = i

    ax.plot(x, y, 'b-', lw=1, alpha=0.6, label='pdf')
    ax.set_ylabel(y_label, fontsize=11)
    ax.set_xlabel(x_label, fontsize=11)
    fig.suptitle(title)
    #plt.legend(loc='upper right', fontsize= 12)
    plt.show()

In [ ]:
plot_trace(mh_simulation["mu"], r'iteration', r'$\mu$', f'Markov chain with length {N}')

The histogram of the chain path is shown below. Notably, this path produces a remarkably accurate approximation of the $N(4,0.62)$ posterior. Through a rigorous and formal process, we utilized dependent draws from a Uniform model to approximate a Normal model. And it worked!

In [ ]:
mean    = 4.0
std_dev = 0.62

fig, ax = plt.subplots(1, 1)

# Display the probability density function N(4,0.62)

x = np.linspace(mean-3.5*std_dev,mean+3.5*std_dev, 100)
ax.plot(x, norm.pdf(x, loc=mean, scale=std_dev), 'r-', lw=5, alpha=0.6, label='pdf')

# Plot the markov chain histogram

ax.set_xlim([x[0], x[-1]])
sns.histplot(ax=ax, data=mh_simulation["mu"], stat='density', bins=20, discrete=False, kde=False,
            facecolor='#888888', edgecolor='#AAAAAA', label='sample hist.')
ax.set_ylabel('f($\mu \mid y$)', fontsize=11)
ax.set_xlabel(r'$\mu$', fontsize=11)
fig.suptitle(f'Markov chain approximation of the posterior N($\mu$={mean},$\sigma$={std_dev})')
plt.legend(loc='upper right', fontsize= 12)
plt.show()

## 7.4 Tuning the Metropolis-Hastings algorithm

Let's wade into one nal weed patch. In implementing the Metropolis-Hastings algorithm above, we utilized a Uniform proposal model $\mu' \mid \mu \sim Uniform(\mu-w,\mu+w)$ with half-width $w$=1. Our selection of $w$=1 defined the neighborhood or range of potential proposals (Figure 7.5). Naturally, the choice of $w$ impacts the performance of our Markov chain tour. Check the intuition about $w$ with the followin quiz.

**Quiz**

Figure 7.8 presents trace plots and histograms of three separate Metropolis-Hastings tours of the $N(4,0.62)$ posterior. Each tour utilizes a Uniform proposal model, but with different half-widths: $w$=0.01, $w$=1, or $w$=100. Match each tour to the w with which it was generated.

![](../fig/fig-7-08-mcmc-metropolis-hastings3.png) <P>
_Figure 7.8_

**Answer**: Tour 1 uses $w$=100, Tour 2 uses $w$=0.01, and Tour 3 uses $w$=1.

The main conclusion is that the Metropolis-Hastings algorithm can work - we have seen as much - but we have to tune it. In our example, this means that we have to pick an appropriate half-width $w$ for the Uniform proposal model. The tours in the quiz above illustrate the challenge this represents: we do not want $w$ to be too small or too large, but just right. Tour 2 illustrates what can go wrong when $w$ is too small (here $w$=0.01). You can reproduce these results with the following code.

In [ ]:
mh_simulation = mh_path(N=5000, w=0.01)
plot_trace(mh_simulation["mu"], r'iteration', r'$\mu$', f'Markov chain with length {N} (w=00.1)')

In this case, the Uniform proposal model places a narrow neighborhood around the chain current location – the chain can only move within 0.01 units at a time. Proposals will therefore tend to be very close to the current location, and thus
have similar posterior plausibility and a high probability of being accepted. Specifically, when a proposal $\mu' \approx \mu$, it is typically the case that $f(\mu')\times L(\mu' \mid y) \approx f(\mu) \times L(\mu \mid y)$ hence

$\alpha = min \{ 1, \frac{f(\mu') \times L(\mu' \mid y)}{f(\mu) \times L(\mu \mid y)} \} \approx min\{1,1\} = 1$

The result is a Markov chain that is almost always moving, but takes very small steps that it will take a very long time to explore the entire posterior plausible region of $\mu$ values. Tour 1 illustrates the other extreme in which $w$ is too large (here $w$=100).

In [ ]:
mh_simulation = mh_path(N=5000, w=100)
plot_trace(mh_simulation["mu"], r'iteration', r'$\mu$', f'Markov chain with length {N} (w=100)')

By utilizing a Uniform proposal model with a very large neighborhood around the chain current location, proposals can fall outside the region of posterior plausible $\mu$. In this case, proposals will often be rejected, resulting in a path which gets stuck at the same location for multiple iterations in a row (as evidenced by the at parts of the trace plot). Tour 3 represents a correct choice for $w$. This is our original tour, which utilized $w$=1, a neighborhood size which is neither too small nor too big but just right. The corresponding tour efficiently explores the posterior plausible region of $\mu$, not getting stuck in one place or region for too long.

## 7.5 A Beta-Binomial example

For extra practice, let us implement the Metropolis-Hastings algorithm for a Beta-Binomial model in which we observe $Y$=1 success in 2 trials:

$Y \mid \pi \sim Binom(2, \pi)$ (the likelihood)<P>
$\pi \sim Beta(2, 3)$ (the prior for $\pi$<P>

Then, we know that the posterior would follow a Beta ditribution $\pi \mid Y=1 \sim Beta(3, 4)$.

Again, we suppose that we were only able to define the posterior PDFf up to some missing normalizing constant:

$f(\pi \mid y=1) \propto f(\pi) \times L(\pi \mid y=1)$

where $f(\pi)$ is the $Beta(2,3)$ prior PDF and $L(\pi \mid y=1)$ is the $Binom(2, \pi)$ likelihood function. Our goal then is to construct a Metropolis-Hastings navigation path across the posterior, $\{\pi^{(1)}, \pi^{(3)}, \dots, \pi^{(N)}\}$, utilizing a two-step iterative process: (1) at the current location $\pi$ in our path, take a random draw $\pi'$ from a proposal model; and then (2) decide whether or not to move there. In step 1, the proposed locations would
ideally be restricted to be between 0 and 1, just like $\pi$ itself. Since the Uniform proposal model we used above might propose values outside this range, we will instead tune and utilize a $Beta(a, b)$ proposal model. Further, we will utilize the same $Beta(a, b)$ proposal model at each step in the chain. As such, our proposal strategy does not depend on the current location, but the decision on whether or not we accept a proposal dependd on the current location. This special case of the Metropolis-Hastings is referred to as the **independence sampling algorithm**.

**Independence sampling algorithm**

The independence sampling algorithm is a special case of the Metropolis-Hastings in which the same proposal model is utilized at each iteration, independent of the chain's current location. That is, from a current location $\pi$, a proposal $\pi'$ is drawn from a proposal model with PDF $q(\pi')$ (as opposed to $q(\pi' \mid \pi)$. Thus, the acceptance probability defined in equation 7.3 simplifies to:

$\alpha = min \{ 1, \frac{f(\pi') \times L(\pi' \mid y)}{f(\pi) \times L(\pi \mid y)} \frac{q(\pi)}{q(\pi')} \}$

We can rewrite $\alpha$ for the independence sampler to emphasize its dependence on the relative posterior plausibility of proposal $\pi'$ versus current location $\pi$:

$\alpha = min \{ 1, \frac{f(\pi') \times L(\pi' \mid y)/f(y)}{f(\pi) \times L(\pi \mid y)/f(y)} \frac{q(\pi)}{q(\pi')} \} = = min \{ 1, \frac{f(\pi' \mid y)}{f(\pi \mid y)} \frac{q(\pi)}{q(\pi')}\}$

Like the acceptance probability for the Metropolis algorithm, the previous equation includes the posterior PDF ratio $\frac{f(\pi' \mid y)}{f(\pi \mid y)}$. This ensures that the independence sampler weighs the relative posterior plausibility of $\pi'$ versus $\pi$ in making its moves. Yet unlike the Metropolis acceptance probability, $q(\pi)$ and $q(\pi')$ are not typically equal, and thus do not cancel out of the formula. Rather, the inclusion of their ratio serves as a corrective measure: the probability $\alpha$ of accepting a proposal $\pi'$ decreases as $q(\pi')$ increases. Conceptually, we place a penalty on common proposal values, ensuring that our path does not float toward these values simply because we keep proposing them.

The `one_iteration` function below implements a single iteration of this independence sampling algorithm, starting from any current value $\pi$ and utilizing a $Beta(a, b)$ proposal model for any given $a$ and $b$. In the calculation of
acceptance probability $\alpha$ (`alpha`), notice that we utilize `beta` to evaluate the prior and proposal PDFs as well as `binom` to evaluate the Binomial likelihood function with data $Y$=1, $n$=2, and an unknown probability $\pi$.

In [ ]:
def one_beta_binom_iteration(a, b, current):
    proposal       = rng.beta(a, b) # new value proposed for 'pi' indpendent of its current value
    proposal_p_x_l = beta.pdf(proposal, 2, 3) * binom.pmf(1, 2, proposal) # prior x likelihood at proposed location
    proposal_q     = beta.pdf(proposal, a, b) # probability 'q' of proposed value 
    current_p_x_l  = beta.pdf(current, 2, 3) * binom.pmf(1, 2, current) # prior x likelihood at current location
    current_q      = beta.pdf(current, a, b)  # probability 'q' of current value 
    alpha          = min(1, proposal_p_x_l / current_p_x_l * current_q / proposal_q) # acceptance probability
    # Decide if we accept (with probabily alpha) or reject (with probabily 1-alpha) 
    # the proposed value for 'pi' using a random selection between proposed and current values
    next_stop      = np.random.choice(
        [proposal, current], size=1, replace=True, p=[alpha, 1-alpha]
    )
    next_stop      = next_stop[0]
    data           = {"proposal": proposal, "alpha": alpha, "next_stop": next_stop}
    return data

Now, we will write a `beta_binom_path` function which constructs an $N$-length Markov chain path for any $Beta(a,b)$ proposal model, utilizing `one_iteration` to determine each new location.

In [ ]:
def beta_binom_path(N, a, b):
    current = 0.5
    pi      = np.zeros(N)
    for step in range(N):
        sim      = one_beta_binom_iteration(a=a, b=b, current=current)
        pi[step] = sim["next_stop"]
        current  = sim["next_stop"]
    return pd.DataFrame(pi, columns=["pi"])

In [ ]:
betabin_sim = beta_binom_path(N=5000, a=1, b=1)

We encourage you to try different tunings of the $Beta(a, b)$ proposal model. To keep it simple here, we run a 5000 step navigation across the Beta-Binomial posterior using a $Beta(1, 1)$, i.e., $Uniform(0, 1)$, proposal model. As such, each proposal is equally likely to be anywhere between 0 and 1, no matter the chain's current location. The results are summarized in next figures. The path trace appears to be stable, random, and provides an excellent approximation of the $Beta(3,4)$ posterior model (which in practice we would not have access to for comparison). That is, the Metropolis-Hastings algorithm allowed us to utilize draws from the $Beta(1,1)$ proposal model to approximate our $Beta(3,4)$ posterior.

In [ ]:
plot_trace(betabin_sim["pi"], r'iteration', r'$\pi$', f'Markov chain with length {5000} an proposal model Beta(1,1)')

In [ ]:
a = 3
b = 4

fig, ax = plt.subplots(1, 1)

# Display the probability density function Beta(3,4)

x = np.linspace(0, 1, 100)
ax.plot(x, beta.pdf(x, a=a, b=b), 'r-', lw=5, alpha=0.6, label='pdf')

# Plot the markov chain histogram

ax.set_xlim([x[0], x[-1]])
sns.histplot(ax=ax, data=betabin_sim["pi"], stat='density', bins=32, discrete=False, kde=False,
            facecolor='#888888', edgecolor='#AAAAAA', label='sample hist.')
ax.set_ylabel('f($\pi \mid y$)', fontsize=11)
ax.set_xlabel(r'$\pi$', fontsize=11)
fig.suptitle(f'Markov chain approximation of the posterior Beta(a={a},b={b})')
plt.legend(loc='upper right', fontsize= 12)
plt.show()

In [ ]:
watermark